In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("test").remote("sc://spark-connector-0-driver-svc:15002").getOrCreate()

print("session created")

df_test = spark.createDataFrame([["test", 1]], ['name', 'num'])
print("Initialized test data")


![](https://media.licdn.com/dms/image/v2/D4E0BAQGumdWW8R2giQ/company-logo_200_200/B4EZ87y9pLK4AE-/0/1783414643504/health_data_research_service_logo?e=2147483647&v=beta&t=GxsLjCFo9AmdCTHkDLK622NOIF3hplTcipdZBeJbzPY)

In [ ]:
from datetime import datetime
import pandas as pd

In [ ]:
# Oxford university Hospitals
OUH_AWS_conditions = "https://raw.githubusercontent.com/willmhbrowne/synthetic_cancer_fihr/refs/heads/main/output/aws/conditions.csv"
OUH_AWS_medications = "https://raw.githubusercontent.com/willmhbrowne/synthetic_cancer_fihr/refs/heads/main/output/aws/medication_requests.csv"
OUH_AWS_patients = "https://raw.githubusercontent.com/willmhbrowne/synthetic_cancer_fihr/refs/heads/main/output/aws/patients.csv"

# University Hospitals Sussex
UHS_AZURE_conditions = "https://raw.githubusercontent.com/willmhbrowne/synthetic_cancer_fihr/refs/heads/main/output/azure/conditions.csv"
UHS_AZURE_medications = "https://raw.githubusercontent.com/willmhbrowne/synthetic_cancer_fihr/refs/heads/main/output/azure/medication_requests.csv"
UHS_AZURE_patients = "https://raw.githubusercontent.com/willmhbrowne/synthetic_cancer_fihr/refs/heads/main/output/azure/patients.csv"

# University Hospitals Birmingham
UHB_GCP_conditions = "https://raw.githubusercontent.com/willmhbrowne/synthetic_cancer_fihr/refs/heads/main/output/gcp/conditions.csv"
UHB_GCP_medications = "https://raw.githubusercontent.com/willmhbrowne/synthetic_cancer_fihr/refs/heads/main/output/gcp/medication_requests.csv"
UHB_GCP_patients = "https://raw.githubusercontent.com/willmhbrowne/synthetic_cancer_fihr/refs/heads/main/output/gcp/patients.csv"



In [ ]:
ouh_aws_cond_df = spark.createDataFrame(pd.read_csv(OUH_AWS_conditions))
ouh_aws_meds_df = spark.createDataFrame(pd.read_csv(OUH_AWS_medications))
ouh_aws_pats_df = spark.createDataFrame(pd.read_csv(OUH_AWS_patients))

uhs_azure_cond_df = spark.createDataFrame(pd.read_csv(UHS_AZURE_conditions))
uhs_azure_meds_df = spark.createDataFrame(pd.read_csv(UHS_AZURE_medications))
uhs_azure_pats_df = spark.createDataFrame(pd.read_csv(UHS_AZURE_patients))

uhb_gcp_cond_df = spark.createDataFrame(pd.read_csv(UHB_GCP_conditions))
uhb_gcp_meds_df = spark.createDataFrame(pd.read_csv(UHB_GCP_medications))
uhb_gcp_pats_df = spark.createDataFrame(pd.read_csv(UHB_GCP_patients))




In [ ]:
#
# AWS
#

# discriminator = datetime.now().strftime("%d%B%Y_%H_%M")
aws_namespace = "aws.acceptancetests"
aws_tables = [(ouh_aws_cond_df,"ouh_conditions"),
              (ouh_aws_meds_df,"ouh_medications"),
              (ouh_aws_pats_df,"ouh_patients")]

spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {aws_namespace}")
print(f"Created namespace {aws_namespace}")

for df,t in aws_tables:
    aws_fqn = f"{aws_namespace}.{t}" 
    df.writeTo(aws_fqn).create()
    print(f"Wrote {aws_fqn} to AWS")

# spark.sql(f"SELECT * FROM {aws_fqn}").show()
spark.sql("describe table extended aws.acceptancetests.ouh_conditions").show(truncate=False)
spark.sql("describe table extended aws.acceptancetests.ouh_conditions").filter("col_name = 'Location'").show(truncate=False)
spark.sql("show tblproperties aws.acceptancetests.ouh_conditions").show(truncate=False)

In [ ]:
#
# GCP
#


# discriminator = datetime.now().strftime("%d%B%Y_%H_%M")
gcp_namespace = "gcp.acceptancetests"
gcp_tables = [(uhb_gcp_cond_df,"uhb_conditions"),
              (uhb_gcp_meds_df,"uhb_medications"),
              (uhb_gcp_pats_df,"uhb_patients")]

spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {gcp_namespace}")
print(f"Created namespace {gcp_namespace}")

for df,t in gcp_tables:
    gcp_fqn = f"{gcp_namespace}.{t}" 
    df.writeTo(gcp_fqn).create()
    print(f"Wrote {gcp_fqn} to gcp")


In [ ]:
uhs_azure_meds_df

In [ ]:
#
# azure
#
cond_cols = """condition_id String, patient_id String, snomed_code BIGINT, icd10_code String,
description string, clinical_status string, verification_status string, stage string,
onset_datetime string, recorded_date string"""

meds_cols = """medication_request_id string, patient_id string, 
condition_id string, rxnorm_code bigint, medication string, status string, intent string,
authored_on string, dosage_text string, frequency bigint, period_days bigint"""

pat_cols = """patient_id string, mrn string, family_name string, given_name string, gender string,
birth_date string, cancer_type string"""


# discriminator = datetime.now().strftime("%d%B%Y_%H_%M")
azure_namespace = "azure.acceptancetests"
azure_tables = [(uhs_azure_cond_df,"uhs_conditions",cond_cols),
              (uhs_azure_meds_df,"uhs_medication",meds_cols),
              (uhs_azure_pats_df,"uhs_patients",pat_cols)]


spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {azure_namespace}")
print(f"Created namespace {azure_namespace}")



for df,t,c in azure_tables:
    az_fqn = f"{azure_namespace}.{t}" 
    spark.sql(f"CREATE TABLE IF NOT EXISTS {az_fqn} ({c}) USING iceberg")
    df.writeTo(az_fqn).append()
    print(f"Wrote {azure_fqn} to azure")



In [ ]:
spark.sql("describe table extended aws.acceptancetests.ouh_conditions").show(truncate=False)
spark.sql("describe table extended aws.acceptancetests.ouh_conditions").filter("col_name = 'Location'").show(truncate=False)
spark.sql("show tblproperties aws.acceptancetests.ouh_conditions").show(truncate=False)


In [ ]:
#
# GCP
#
discriminator = datetime.now().strftime("%d%B%Y_%H_%M")
gcp_namespace = "gcp.acceptancetests"
gcp_fqn = f"{gcp_namespace}.smoketesttable_" + discriminator
spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {gcp_namespace}")
print(f"Created namespace {gcp_namespace}")

df_test.writeTo(gcp_fqn).create()
print(f"Wrote {gcp_fqn} to GCP")

df_read = spark.read.table(gcp_fqn)
df_read.show()


In [ ]:
#
# AZURE
#
discriminator = datetime.now().strftime("%d%B%Y_%H_%M")
az_namespace = "azure.acceptancetests"
az_fqn = f"{az_namespace}.smoketesttable_" + discriminator

spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {az_namespace}")
print(f"Created namespace {az_namespace}")

spark.sql(f"CREATE TABLE IF NOT EXISTS {az_fqn} (name String, num Integer) USING iceberg")
df_test.writeTo(az_fqn).append()

df_read = spark.read.table(az_fqn)
df_read.show()



In [ ]:

df_gcp = spark.read.table(gcp_fqn)
df_azure = spark.read.table(az_fqn)
df_aws = spark.read.table(aws_fqn)

df_joined = (
    df_gcp
    .join(df_azure, on="name", how="inner")
    .join(df_aws, on="name", how="inner")
)
df_joined.show()
